# Ngày 30 · Dự án tổng kết: ablation, phân tích lỗi và submission

- Đo đóng góp của từng nhóm đặc trưng bằng ablation (bỏ từng nhóm, đo lại CV)
- Tìm chỗ mô hình sai nhiều nhất theo lát cắt dữ liệu
- Kiểm tra file submission trước khi nộp, rồi mới mở holdout đúng một lần

**Đọc:**
- Lipton & Steinhardt (2019), Troubling Trends in Machine Learning Scholarship, ACM Queue 17(1) (mục về ablation)
- Ng (2018), Machine Learning Yearning, các chương về phân tích lỗi (error analysis)
- Sculley et al. (2015), Hidden Technical Debt in Machine Learning Systems, NeurIPS 28
- Recht, Roelofs, Schmidt & Shankar (2019), Do ImageNet Classifiers Generalize to ImageNet?, ICML

In [ ]:
import hashlib, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if (p / 'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook từ thư mục py30 hoặc py30/bai_lam để thấy data/real_estate.csv.')
# Real estate valuation (Yeh, 2018; UCI, CC BY 4.0): 414 giao dịch, giá theo 10.000 TWD/ping.
real_estate = pd.read_csv(DATA / 'real_estate.csv')
TARGET = 'price_10k_ntd_per_ping'

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

def _cv_rmse_ref(make_model, X, y, seed):
    out = []
    for tr, va in KFold(5, shuffle=True, random_state=seed).split(X):
        m = make_model().fit(X.iloc[tr], y[tr]); out.append(np.sqrt(np.mean((m.predict(X.iloc[va]) - y[va]) ** 2)))
    return float(np.mean(out))
# Tập phát triển cố định (80%) — 20% còn lại là holdout đã khóa ở Ngày 29.
_order = np.random.default_rng(0).permutation(len(real_estate))
_hold = np.sort(_order[:83]); _dev = np.sort(_order[83:])
dev = real_estate.iloc[_dev].reset_index(drop=True)
X_dev, y_dev = dev.drop(columns=TARGET), dev[TARGET].to_numpy()
from sklearn.model_selection import cross_val_predict
GROUPS = {'vị trí': ['latitude', 'longitude'], 'tiện ích': ['mrt_distance_m', 'stores'], 'tuổi nhà': ['age_years'], 'thời gian': ['transaction_date']}
_gb = lambda: GradientBoostingRegressor(random_state=0)
cv_rmse = globals().get('cv_rmse') or _cv_rmse_ref
_pred30 = cross_val_predict(_gb(), X_dev, y_dev, cv=KFold(5, shuffle=True, random_state=0))
_ids30 = pd.Series(np.arange(1461, 1471), name='Id')
_good30 = pd.DataFrame({'Id': _ids30, 'SalePrice': np.linspace(120000, 300000, 10)})


## Bảng ablation theo nhóm đặc trưng

Gợi ý: full = cv_rmse(make_model, X, y, seed). Với mỗi nhóm: X.drop(columns=cột của nhóm), đo lại, delta = rmse_khi_bỏ − full (dương nghĩa là nhóm có ích). Trả DataFrame cột ["nhóm", "rmse", "delta"], sắp delta giảm dần.

In [ ]:
def ablation(make_model, X, y, groups, seed):
    # groups: dict tên nhóm → list cột. Dùng cv_rmse (đã có sẵn).
    raise NotImplementedError


In [ ]:
_c30 = {}
def _abl():
    if 't' not in _c30:
        _c30['t'] = ablation(_gb, X_dev, y_dev, GROUPS, 0)
    return _c30['t']
def _abl_ref():
    full = _cv_rmse_ref(_gb, X_dev, y_dev, 0)
    return {g: _cv_rmse_ref(_gb, X_dev.drop(columns=c), y_dev, 0) - full for g, c in GROUPS.items()}
_report('d30-ablation',
    _check('Đúng cột và một dòng cho mỗi nhóm', lambda: ((lambda t: list(t.columns) == ['nhóm', 'rmse', 'delta'] and sorted(t['nhóm']) == sorted(GROUPS))(_abl()), '4 nhóm.')),
    _check('delta khớp cách tính tham chiếu', lambda: ((lambda t, r: all(np.isclose(d, r[g]) for g, d in zip(t['nhóm'], t['delta'])))(_abl(), _abl_ref()), 'delta = RMSE khi bỏ nhóm − RMSE đầy đủ.')),
    _check('Sắp xếp theo delta giảm dần', lambda: (bool(np.all(np.diff(_abl()['delta'].to_numpy()) <= 0)), 'Nhóm quan trọng nhất ở trên cùng.')),
)


## Lỗi theo lát cắt và các ca sai nhiều nhất

Gợi ý: Nếu q được cho, chia cột bằng pd.qcut(df[by], q); ngược lại nhóm theo giá trị của cột. Với mỗi nhóm: n, mae = mean|p − y|, bias = mean(p − y). Sắp mae giảm dần. worst_cases: np.argsort(−|p − y|)[:k].

In [ ]:
def error_slices(df, y_true, y_pred, by, q=None):
    # Trả DataFrame index = nhóm, cột ['n', 'mae', 'bias'], sắp theo mae giảm dần.
    raise NotImplementedError

def worst_cases(y_true, y_pred, k):
    # Chỉ số của k mẫu có |sai số| lớn nhất, lớn nhất đứng đầu.
    raise NotImplementedError


In [ ]:
def _slices_ref(by, q):
    key = pd.qcut(X_dev[by], q) if q else X_dev[by]
    e = _pred30 - y_dev
    return pd.DataFrame({'g': key.to_numpy(), 'a': np.abs(e), 'e': e}).groupby('g', observed=True).agg(n=('a', 'size'), mae=('a', 'mean'), bias=('e', 'mean')).sort_values('mae', ascending=False)
def _same(a, b):
    return list(a.columns) == ['n', 'mae', 'bias'] and len(a) == len(b) and np.allclose(a[['n', 'mae', 'bias']].to_numpy(float), b.to_numpy(float))
_report('d30-errors',
    _check('Lát cắt theo tứ phân vị khoảng cách MRT', lambda: (_same(error_slices(X_dev, y_dev, _pred30, 'mrt_distance_m', 4), _slices_ref('mrt_distance_m', 4)), 'pd.qcut thành 4 nhóm.')),
    _check('Lát cắt theo giá trị rời rạc (số cửa hàng)', lambda: (_same(error_slices(X_dev, y_dev, _pred30, 'stores'), _slices_ref('stores', None)), 'Mỗi giá trị stores là một nhóm.')),
    _check('Bias có dấu đúng (dự đoán − thật)', lambda: (np.isclose(error_slices(pd.DataFrame({'g': ['a', 'a']}), np.array([10., 10.]), np.array([12., 14.]), 'g').loc['a', 'bias'], 3.0), 'Dự đoán cao hơn thật thì bias dương.')),
    _check('worst_cases trả đúng thứ tự', lambda: (list(worst_cases(y_dev, _pred30, 3)) == list(np.argsort(-np.abs(_pred30 - y_dev), kind='stable')[:3]), 'Ca sai nhiều nhất đứng đầu.')),
)


## Kiểm tra file submission trước khi nộp

Gợi ý: Kiểm tra lần lượt và thêm một câu mô tả vào danh sách cho mỗi lỗi: đúng hai cột [id_col, target_col] theo thứ tự; số dòng bằng số id test; cột id trùng khớp từng dòng với test_ids; không có NaN; cột mục tiêu là số và dương.

In [ ]:
def check_submission(sub, test_ids, id_col='Id', target_col='SalePrice'):
    # Trả list các chuỗi mô tả lỗi; list rỗng nghĩa là hợp lệ.
    raise NotImplementedError


In [ ]:
_bad30 = {
    'thiếu cột': _good30[['Id']],
    'sai tên cột': _good30.rename(columns={'SalePrice': 'price'}),
    'thiếu dòng': _good30.iloc[:9],
    'sai thứ tự id': _good30.iloc[::-1].reset_index(drop=True),
    'có NaN': _good30.assign(SalePrice=[np.nan] + list(_good30.SalePrice[1:])),
    'giá âm': _good30.assign(SalePrice=[-1.] + list(_good30.SalePrice[1:])),
    'giá là chữ': _good30.assign(SalePrice=_good30.SalePrice.astype(str)),
}
_report('d30-submission',
    _check('File đúng thì không báo lỗi', lambda: (check_submission(_good30, _ids30) == [], '10 dòng, Id 1461–1470.')),
    _check('Bắt được cả 7 kiểu lỗi', lambda: ((lambda miss: (not miss, 'Bỏ sót: ' + ', '.join(miss) if miss else 'Đủ 7.'))([k for k, v in _bad30.items() if not check_submission(v, _ids30)]))),
)


## Dự án

Trên House Prices: (1) bảng ablation bằng RMSLE 5-fold với các nhóm đặc trưng bạn tạo (diện tích, chất lượng, vị trí, năm xây...), (2) bảng lỗi theo lát cắt (Neighborhood, OverallQual) và 10 nhà sai nhiều nhất kèm giả thuyết, (3) chạy check_submission rồi nộp bản cuối. Cuối cùng mở holdout đúng MỘT lần, báo cáo RMSLE holdout cạnh điểm CV và điểm Kaggle. Viết báo cáo 1 trang: bài toán, baseline, mô hình cuối, ablation, phân tích lỗi, hạn chế.